# pandas 시계열(Time Series) 튜토리얼 50제
> 참조: https://pandas.pydata.org/docs/user_guide/timeseries.html
> 데이터: classicmodels 샘플 CSV (Google Drive 마운트)

---

## 목차

### Part 1. Timestamp & 날짜 생성 (Q01 ~ Q10)
### Part 2. .dt 접근자 — 시분초 분해 (Q11 ~ Q20)
### Part 3. 시간 차이 & Timedelta (Q21 ~ Q28)
### Part 4. DateOffset & 비즈니스 날짜 (Q29 ~ Q33)
### Part 5. 인덱싱 & 슬라이싱 (Q34 ~ Q38)
### Part 6. Resample & 집계 (Q39 ~ Q44)
### Part 7. 시프트 & 롤링 (Q45 ~ Q48)
### Part 8. 타임존 (Q49 ~ Q50)


---
## 0. 환경 설정

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import datetime

pd.set_option("display.max_columns", 15)
pd.set_option("display.width", 110)

# Google Drive 경로 (업로드한 폴더로 수정)
DATA = "/content/drive/MyDrive/Colab Notebooks/2026/이어드림스쿨6기/dataset/erd_sample"

orders       = pd.read_csv(f"{DATA}/orders.csv",
                           parse_dates=["orderDate", "requiredDate", "shippedDate"])
payments     = pd.read_csv(f"{DATA}/payments.csv",
                           parse_dates=["paymentDate"])
orderdetails = pd.read_csv(f"{DATA}/orderdetails.csv")
customers    = pd.read_csv(f"{DATA}/customers.csv")

print("로드 완료")
print(orders.dtypes[["orderDate","requiredDate","shippedDate"]])


로드 완료
orderDate       datetime64[ns]
requiredDate    datetime64[ns]
shippedDate     datetime64[ns]
dtype: object


---
## Part 1. Timestamp & 날짜 생성

> `pd.Timestamp`, `pd.to_datetime`, `pd.date_range`, `pd.period_range`


### Q01. 문자열 -> Timestamp 변환 (3가지 방법)

In [ ]:
#

2003-01-06 00:00:00 <class 'pandas._libs.tslibs.timestamps.Timestamp'>
2003-01-06 00:00:00 <class 'pandas._libs.tslibs.timestamps.Timestamp'>
2003-01-06 00:00:00 <class 'pandas._libs.tslibs.timestamps.Timestamp'>
세 값이 같은가? True


### Q02. 다양한 형식의 날짜 문자열을 한번에 변환

In [ ]:
#

DatetimeIndex(['2003-01-06', '2003-01-09', '2003-01-10', '2003-10-01'], dtype='datetime64[ns]', freq=None)
dtype: datetime64[ns]


### Q03. 잘못된 날짜 처리 — errors='coerce'

In [ ]:
#

DatetimeIndex(['2003-01-06', '2003-01-09', '2003-01-10', '2003-01-29', '2003-01-31', '2003-02-11',
               '2003-02-17', '2003-02-24', '2003-03-03', '2003-03-10',        'NaT',        'NaT'],
              dtype='datetime64[ns]', freq=None)
NaT 개수: 2


### Q04. Unix epoch(초) -> Timestamp 변환

In [ ]:
#

epoch 초:
0    1041811200
1    1042070400
2    1042156800
3    1043798400
4    1043971200
Name: orderDate, dtype: int64

복원된 날짜:
0   2003-01-06
1   2003-01-09
2   2003-01-10
3   2003-01-29
4   2003-01-31
Name: orderDate, dtype: datetime64[ns]


### Q05. 여러 컬럼(year, month, day)으로 날짜 조립

In [ ]:
#

0   2003-01-06 09:00:00
1   2003-02-14 13:30:00
2   2003-03-03 17:45:00
dtype: datetime64[ns]


### Q06. date_range — 일별 날짜 시퀀스 생성

In [ ]:
#

총 90일
DatetimeIndex(['2003-01-01', '2003-01-02', '2003-01-03', '2003-01-04', '2003-01-05'], dtype='datetime64[ns]', freq='D') ... DatetimeIndex(['2003-03-29', '2003-03-30', '2003-03-31'], dtype='datetime64[ns]', freq='D')


### Q07. date_range — 다양한 freq 옵션 비교

In [ ]:
#

freq=W   (주별): [Timestamp('2003-01-05 00:00:00'), Timestamp('2003-01-12 00:00:00'), Timestamp('2003-01-19 00:00:00'), Timestamp('2003-01-26 00:00:00'), Timestamp('2003-02-02 00:00:00'), Timestamp('2003-02-09 00:00:00')]

freq=ME  (월말): [Timestamp('2003-01-31 00:00:00'), Timestamp('2003-02-28 00:00:00'), Timestamp('2003-03-31 00:00:00'), Timestamp('2003-04-30 00:00:00'), Timestamp('2003-05-31 00:00:00'), Timestamp('2003-06-30 00:00:00')]

freq=QE  (분기말): [Timestamp('2003-03-31 00:00:00'), Timestamp('2003-06-30 00:00:00'), Timestamp('2003-09-30 00:00:00'), Timestamp('2003-12-31 00:00:00'), Timestamp('2004-03-31 00:00:00'), Timestamp('2004-06-30 00:00:00')]

freq=h   (시간): [Timestamp('2003-01-01 00:00:00'), Timestamp('2003-01-01 01:00:00'), Timestamp('2003-01-01 02:00:00'), Timestamp('2003-01-01 03:00:00'), Timestamp('2003-01-01 04:00:00'), Timestamp('2003-01-01 05:00:00')]



### Q08. bdate_range — 영업일 기준 날짜 생성

In [ ]:
#

영업일: 23일 (주말 제외)
DatetimeIndex(['2003-01-01', '2003-01-02', '2003-01-03', '2003-01-06', '2003-01-07', '2003-01-08',
               '2003-01-09', '2003-01-10', '2003-01-13', '2003-01-14', '2003-01-15', '2003-01-16',
               '2003-01-17', '2003-01-20', '2003-01-21', '2003-01-22', '2003-01-23', '2003-01-24',
               '2003-01-27', '2003-01-28', '2003-01-29', '2003-01-30', '2003-01-31'],
              dtype='datetime64[ns]', freq='B')


### Q09. period_range — 월별 Period 시퀀스

In [ ]:
#

PeriodIndex(['2003-01', '2003-02', '2003-03', '2003-04', '2003-05', '2003-06'], dtype='period[M]')
dtype: period[M]

각 Period의 시작일:
DatetimeIndex(['2003-01-01', '2003-02-01', '2003-03-01', '2003-04-01', '2003-05-01', '2003-06-01'], dtype='datetime64[ns]', freq='MS')

각 Period의 종료일:
DatetimeIndex(['2003-01-31 23:59:59.999999999', '2003-02-28 23:59:59.999999999',
               '2003-03-31 23:59:59.999999999', '2003-04-30 23:59:59.999999999',
               '2003-05-31 23:59:59.999999999', '2003-06-30 23:59:59.999999999'],
              dtype='datetime64[ns]', freq=None)


### Q10. Timestamp 속성 탐색 — year, quarter, is_leap_year 등

In [ ]:
#

날짜      : 2003-01-06
연도      : 2003
월        : 1
일        : 6
요일 이름 : Monday
분기      : Q1
윤년 여부 : False
주 번호   : 2
연중 몇째 날 : 6


---
## Part 2. .dt 접근자 — 시분초 분해

> Series 에 datetime dtype 이 있으면 `.dt` 로 날짜 구성 요소를 추출할 수 있다.


### Q11. .dt.year / .dt.month / .dt.day 추출

In [ ]:
#

,orderDate,year,month,day
0,2003-01-06,2003,1,6
1,2003-01-09,2003,1,9
2,2003-01-10,2003,1,10
3,2003-01-29,2003,1,29
4,2003-01-31,2003,1,31


### Q12. .dt.hour / .dt.minute / .dt.second 추출 (시분초)

In [ ]:
#

0   2003-01-06 09:05:30
1   2003-01-06 10:35:30
2   2003-01-06 12:05:30
3   2003-01-06 13:35:30
4   2003-01-06 15:05:30
dtype: datetime64[ns]

hour  : [9, 10, 12, 13, 15]
minute: [5, 35, 5, 35, 5]
second: [30, 30, 30, 30, 30]


### Q13. .dt.day_name() — 요일 이름 추출

In [ ]:
#

   orderDate    weekday
0 2003-01-06     Monday
1 2003-01-09   Thursday
2 2003-01-10     Friday
3 2003-01-29  Wednesday
4 2003-01-31     Friday
5 2003-02-11    Tuesday
6 2003-02-17     Monday
7 2003-02-24     Monday
8 2003-03-03     Monday
9 2003-03-10     Monday

요일별 주문 건수:
weekday
Monday       5
Friday       2
Thursday     1
Wednesday    1
Tuesday      1
Name: count, dtype: int64


### Q14. .dt.dayofweek — 요일 번호 (월=0 ~ 일=6)

In [ ]:
#

   orderDate  dow  is_weekend
0 2003-01-06    0       False
1 2003-01-09    3       False
2 2003-01-10    4       False
3 2003-01-29    2       False
4 2003-01-31    4       False
5 2003-02-11    1       False
6 2003-02-17    0       False
7 2003-02-24    0       False
8 2003-03-03    0       False
9 2003-03-10    0       False

주말 주문 건수: 0


### Q15. .dt.quarter — 분기 추출

In [ ]:
#

   orderDate  quarter
0 2003-01-06        1
1 2003-01-09        1
2 2003-01-10        1
3 2003-01-29        1
4 2003-01-31        1
5 2003-02-11        1
6 2003-02-17        1
7 2003-02-24        1
8 2003-03-03        1
9 2003-03-10        1

분기별 주문 건수:
quarter
1    10
Name: orderNumber, dtype: int64


### Q16. .dt.week / .dt.isocalendar() — ISO 주 번호

In [ ]:
#

   year  week  day
0  2003     2    1
1  2003     2    4
2  2003     2    5
3  2003     5    3
4  2003     5    5
5  2003     7    2
6  2003     8    1
7  2003     9    1
8  2003    10    1
9  2003    11    1

주차별 주문 건수:
iso_week
2     3
5     2
7     1
8     1
9     1
10    1
11    1
Name: orderNumber, dtype: int64


### Q17. .dt.days_in_month — 해당 월의 총 일수

In [ ]:
#

,paymentDate,days_in_month,is_last_day
0,2004-10-19,31,False
1,2003-06-05,30,False
2,2003-06-06,30,False
3,2003-05-20,31,False
4,2004-02-02,29,False
5,2004-11-09,30,False
6,2004-12-18,31,False
7,2003-02-14,28,False
8,2003-03-15,31,False
9,2004-07-09,31,False


### Q18. .dt.normalize() — 시간 부분을 00:00:00 으로 초기화

In [ ]:
#

원본:
0   2003-01-06 09:05:30
1   2003-01-06 10:35:30
2   2003-01-06 12:05:30
3   2003-01-06 13:35:30
dtype: datetime64[ns]

normalize (시간 제거):
0   2003-01-06
1   2003-01-06
2   2003-01-06
3   2003-01-06
dtype: datetime64[ns]


### Q19. .dt.strftime() — 날짜 -> 원하는 문자열 포맷

In [ ]:
#

,orderDate,date_str_kr,date_str_us
0,2003-01-06,2003년 01월 06일 (Mon),"January 06, 2003"
1,2003-01-09,2003년 01월 09일 (Thu),"January 09, 2003"
2,2003-01-10,2003년 01월 10일 (Fri),"January 10, 2003"
3,2003-01-29,2003년 01월 29일 (Wed),"January 29, 2003"
4,2003-01-31,2003년 01월 31일 (Fri),"January 31, 2003"
5,2003-02-11,2003년 02월 11일 (Tue),"February 11, 2003"


### Q20. .dt.to_period() — Timestamp -> Period 변환

In [ ]:
#

   orderDate order_month             order_week
0 2003-01-06     2003-01  2003-01-06/2003-01-12
1 2003-01-09     2003-01  2003-01-06/2003-01-12
2 2003-01-10     2003-01  2003-01-06/2003-01-12
3 2003-01-29     2003-01  2003-01-27/2003-02-02
4 2003-01-31     2003-01  2003-01-27/2003-02-02
5 2003-02-11     2003-02  2003-02-10/2003-02-16
6 2003-02-17     2003-02  2003-02-17/2003-02-23
7 2003-02-24     2003-02  2003-02-24/2003-03-02

월별 주문 건수:
order_month
2003-01    5
2003-02    3
2003-03    2
Freq: M, Name: orderNumber, dtype: int64


---
## Part 3. 시간 차이 & Timedelta

> `pd.Timedelta`, `pd.to_timedelta`, `timedelta_range`


### Q21. 두 날짜의 차이 -> Timedelta

In [ ]:
#

   orderDate shippedDate lead_days
0 2003-01-06  2003-01-10    4 days
1 2003-01-09  2003-01-11    2 days
2 2003-01-10  2003-01-14    4 days
3 2003-01-29  2003-02-02    4 days
4 2003-01-31  2003-01-29   -2 days
5 2003-02-11  2003-02-12    1 days
6 2003-02-17  2003-02-21    4 days
7 2003-02-24  2003-02-26    2 days
8 2003-03-03  2003-03-08    5 days
9 2003-03-10  2003-03-11    1 days

타입: timedelta64[ns]


### Q22. Timedelta에서 일수(int) 추출 — .dt.days

In [ ]:
#

   orderDate shippedDate lead_days  lead_days_int
0 2003-01-06  2003-01-10    4 days              4
1 2003-01-09  2003-01-11    2 days              2
2 2003-01-10  2003-01-14    4 days              4
3 2003-01-29  2003-02-02    4 days              4
4 2003-01-31  2003-01-29   -2 days             -2
5 2003-02-11  2003-02-12    1 days              1
6 2003-02-17  2003-02-21    4 days              4
7 2003-02-24  2003-02-26    2 days              2
8 2003-03-03  2003-03-08    5 days              5
9 2003-03-10  2003-03-11    1 days              1

평균 리드타임: 2.5 일
최대 리드타임: 5 일
최소 리드타임: -2 일


### Q23. 납기 초과 주문 찾기 (requiredDate vs shippedDate)

In [ ]:
#

납기 초과 주문: 0건
Empty DataFrame
Columns: [orderNumber, requiredDate, shippedDate, overdue_days]
Index: []


### Q24. pd.Timedelta 객체 생성 — 다양한 표현 방법

In [ ]:
#

5 days 00:00:00 5 days 00:00:00 5 days 00:00:00 5 days 00:00:00
모두 같은가? True

2일 3시간 30분 15초: 2 days 03:30:15
  총 초: 185415


### Q25. Timedelta 연산 — 날짜에 더하기/빼기

In [ ]:
#

기준일           : 2003-01-06 00:00:00
+ 7일            : 2003-01-13 00:00:00
+ 2주            : 2003-01-20 00:00:00
- 1달(30일 근사) : 2002-12-07 00:00:00
+ 6시간          : 2003-01-06 06:00:00


,orderDate,expected_ship
0,2003-01-06,2003-01-09
1,2003-01-09,2003-01-12
2,2003-01-10,2003-01-13
3,2003-01-29,2003-02-01
4,2003-01-31,2003-02-03


### Q26. timedelta_range — 일정 간격 Timedelta 시퀀스

In [ ]:
#

TimedeltaIndex(['0 days', '2 days', '4 days', '6 days', '8 days', '10 days'], dtype='timedelta64[ns]', freq='2D')

총 초: [0.0, 172800.0, 345600.0, 518400.0, 691200.0, 864000.0]


### Q27. 시간 차이 분해 — components 속성

In [ ]:
#

Timedelta 구성 요소:
   days  hours  minutes  seconds  milliseconds  microseconds  nanoseconds
0     4      0        0        0             0             0            0
1     2      0        0        0             0             0            0
2     4      0        0        0             0             0            0
3     4      0        0        0             0             0            0
4    -2      0        0        0             0             0            0
5     1      0        0        0             0             0            0
6     4      0        0        0             0             0            0
7     2      0        0        0             0             0            0
8     5      0        0        0             0             0            0
9     1      0        0        0             0             0            0

컬럼: ['days', 'hours', 'minutes', 'seconds', 'milliseconds', 'microseconds', 'nanoseconds']


### Q28. 영업일 기준 경과일 계산 — np.busday_count

In [ ]:
#

   orderDate shippedDate  lead_days_int  lead_bdays
0 2003-01-06  2003-01-10              4           4
1 2003-01-09  2003-01-11              2           2
2 2003-01-10  2003-01-14              4           2
3 2003-01-29  2003-02-02              4           3
4 2003-01-31  2003-01-29             -2          -2
5 2003-02-11  2003-02-12              1           1
6 2003-02-17  2003-02-21              4           4
7 2003-02-24  2003-02-26              2           2
8 2003-03-03  2003-03-08              5           5
9 2003-03-10  2003-03-11              1           1

평균 영업일 리드타임: 2.2 일


---
## Part 4. DateOffset & 비즈니스 날짜

> `pd.offsets.*`, `pd.tseries.offsets.*`


### Q29. BusinessDay — 영업일 기준 날짜 이동

In [ ]:
#

기준일 (금)       : 2003-01-03 00:00:00 Friday
+ 1 영업일 (월)   : 2003-01-06 00:00:00 Monday
+ 5 영업일 (다음주 금): 2003-01-10 00:00:00


,orderDate,next_bday
0,2003-01-06,2003-01-07
1,2003-01-09,2003-01-10
2,2003-01-10,2003-01-13
3,2003-01-29,2003-01-30
4,2003-01-31,2003-02-03


### Q30. MonthEnd / MonthBegin — 월말/월초 이동

In [ ]:
#

기준일          : 2003-01-06 00:00:00
이번 달 말      : 2003-01-31 00:00:00
다음 달 말      : 2003-01-31 00:00:00
이번 달 초      : 2003-01-01 00:00:00
다음 달 초      : 2003-02-01 00:00:00


### Q31. QuarterEnd — 분기 말 이동

In [ ]:
#

Q1 말: 2003-03-31
Q2 말: 2003-06-30
Q3 말: 2003-09-30
Q4 말: 2003-12-31


### Q32. 커스텀 영업일 — 공휴일 제외

In [ ]:
#

기준일 (토): 2002-12-28 00:00:00 Saturday
+ 1 KR영업일: 2002-12-30 00:00:00
신정 다음 한국 영업일: 2003-01-02 00:00:00


### Q33. DateOffset으로 상대적 날짜 계산 — relativedelta 스타일

In [ ]:
#

기준일           : 2003-01-31 00:00:00
+ 1달            : 2003-02-28 00:00:00
+ 1년 2달        : 2004-03-31 00:00:00
+ 1년 3달 5일    : 2004-05-05 00:00:00


---
## Part 5. 인덱싱 & 슬라이싱

> DatetimeIndex 를 인덱스로 사용하면 부분 문자열로 쉽게 필터링 가능


### Q34. DatetimeIndex 설정 & 날짜로 행 선택

In [ ]:
#

인덱스 타입: <class 'pandas.core.indexes.datetimes.DatetimeIndex'>

2003-01-06 주문:
orderNumber                       10100
requiredDate        2003-01-13 00:00:00
shippedDate         2003-01-10 00:00:00
status                          Shipped
comments                            NaN
customerNumber                      363
year                               2003
month                                 1
day                                   6
weekday                          Monday
dow                                   0
is_weekend                        False
quarter                               1
iso_week                              2
date_str_kr         2003년 01월 06일 (Mon)
date_str_us            January 06, 2003
order_month                     2003-01
order_week        2003-01-06/2003-01-12
lead_days               4 days 00:00:00
lead_days_int                         4
overdue_days                         -3
expected_ship       2003-01-09 00:00:00
next_bday           2003-01-07 00:00:00
Na

### Q35. 부분 문자열 슬라이싱 — 연도/월로 필터

In [ ]:
#

2003년 주문 건수: 10
2003년 1월 주문: 5
            orderNumber shippedDate   status
orderDate                                   
2003-01-06        10100  2003-01-10  Shipped
2003-01-09        10101  2003-01-11  Shipped
2003-01-10        10102  2003-01-14  Shipped
2003-01-29        10103  2003-02-02  Shipped
2003-01-31        10104  2003-01-29  Shipped


### Q36. 날짜 범위 슬라이싱

In [ ]:
#

2003-01 ~ 02 주문: 8건
            orderNumber shippedDate
orderDate                          
2003-01-06        10100  2003-01-10
2003-01-09        10101  2003-01-11
2003-01-10        10102  2003-01-14
2003-01-29        10103  2003-02-02
2003-01-31        10104  2003-01-29


### Q37. boolean 마스크로 시간 필터링

In [ ]:
#

2003 Q1 주문: 10건
월요일 주문: 5건


### Q38. truncate() — 날짜 기준 앞뒤 잘라내기

In [ ]:
#

2003-02-01 이후: 5건
2003-02-28 이전: 8건


---
## Part 6. Resample & 집계

> `resample()` 은 시계열 데이터를 다른 주기로 집계하는 핵심 도구


### Q39. resample('W') — 주별 주문 건수 집계

In [ ]:
#

orderDate
2003-01-12    3
2003-01-19    0
2003-01-26    0
2003-02-02    2
2003-02-09    0
2003-02-16    1
2003-02-23    1
2003-03-02    1
2003-03-09    1
2003-03-16    1
Freq: W-SUN, Name: order_count, dtype: int64


### Q40. resample('ME') — 월별 결제 금액 합계

In [ ]:
#

                total       avg  txn_count
paymentDate                               
2003-02-28   14191.12  14191.12          1
2003-03-31   20009.53  20009.53          1
2003-04-30       0.00       NaN          0
2003-05-31    6066.78   6066.78          1
2003-06-30   47213.42  23606.71          2
2003-07-31       0.00       NaN          0
2003-08-31       0.00       NaN          0
2003-09-30       0.00       NaN          0
2003-10-31       0.00       NaN          0
2003-11-30       0.00       NaN          0
2003-12-31       0.00       NaN          0
2004-01-31       0.00       NaN          0
2004-02-29   33347.88  33347.88          1
2004-03-31       0.00       NaN          0
2004-04-30       0.00       NaN          0
2004-05-31       0.00       NaN          0
2004-06-30       0.00       NaN          0
2004-07-31   10223.83  10223.83          1
2004-08-31       0.00       NaN          0
2004-09-30       0.00       NaN          0
2004-10-31    6066.78   6066.78          1
2004-11-30 

### Q41. resample('QE') — 분기별 결제 요약

In [ ]:
#

                total           avg  count
paymentDate                               
2003-03-31   34200.65  17100.325000      2
2003-06-30   53280.20  17760.066667      3
2003-09-30       0.00           NaN      0
2003-12-31       0.00           NaN      0
2004-03-31   33347.88  33347.880000      1
2004-06-30       0.00           NaN      0
2004-09-30   10223.83  10223.830000      1
2004-12-31   87748.62  29249.540000      3


### Q42. resample + interpolate — 결측 구간 보간

In [ ]:
#

원본 (처음 10일):
paymentDate
2003-02-14    14191.12
2003-02-15        0.00
2003-02-16        0.00
2003-02-17        0.00
2003-02-18        0.00
2003-02-19        0.00
2003-02-20        0.00
2003-02-21        0.00
2003-02-22        0.00
2003-02-23        0.00
Freq: D, Name: amount, dtype: float64

보간 후 (처음 10일):
paymentDate
2003-02-14    14191.120000
2003-02-15    14391.754828
2003-02-16    14592.389655
2003-02-17    14793.024483
2003-02-18    14993.659310
2003-02-19    15194.294138
2003-02-20    15394.928966
2003-02-21    15595.563793
2003-02-22    15796.198621
2003-02-23    15996.833448
Freq: D, Name: amount, dtype: float64


### Q43. resample('ME').ohlc() — 시가/고가/저가/종가 스타일 집계

In [ ]:
#

                 open      high       low     close
paymentDate                                        
2003-02-28   14191.12  14191.12  14191.12  14191.12
2003-03-31   20009.53  20009.53  20009.53  20009.53
2003-04-30        NaN       NaN       NaN       NaN
2003-05-31    6066.78   6066.78   6066.78   6066.78
2003-06-30   14571.44  32641.98  14571.44  32641.98
2003-07-31        NaN       NaN       NaN       NaN
2003-08-31        NaN       NaN       NaN       NaN
2003-09-30        NaN       NaN       NaN       NaN
2003-10-31        NaN       NaN       NaN       NaN
2003-11-30        NaN       NaN       NaN       NaN
2003-12-31        NaN       NaN       NaN       NaN
2004-01-31        NaN       NaN       NaN       NaN
2004-02-29   33347.88  33347.88  33347.88  33347.88
2004-03-31        NaN       NaN       NaN       NaN
2004-04-30        NaN       NaN       NaN       NaN
2004-05-31        NaN       NaN       NaN       NaN
2004-06-30        NaN       NaN       NaN       NaN
2004-07-31  

### Q44. groupby + Grouper — 연도별 분기별 집계

In [ ]:
#

    year_end quarter_end  order_count
0 2003-12-31  2003-03-31           10


---
## Part 7. 시프트 & 롤링

> `shift()`, `diff()`, `rolling()`, `expanding()`


### Q45. shift() — 전월 대비 결제액 계산

In [ ]:
#

               amount  prev_month  mom_change  mom_pct
paymentDate                                           
2003-02-28   14191.12         NaN         NaN      NaN
2003-03-31   20009.53    14191.12     5818.41     41.0
2003-04-30       0.00    20009.53   -20009.53   -100.0
2003-05-31    6066.78        0.00     6066.78      inf
2003-06-30   47213.42     6066.78    41146.64    678.2
2003-07-31       0.00    47213.42   -47213.42   -100.0
2003-08-31       0.00        0.00        0.00      NaN
2003-09-30       0.00        0.00        0.00      NaN
2003-10-31       0.00        0.00        0.00      NaN
2003-11-30       0.00        0.00        0.00      NaN
2003-12-31       0.00        0.00        0.00      NaN
2004-01-31       0.00        0.00        0.00      NaN
2004-02-29   33347.88        0.00    33347.88      inf
2004-03-31       0.00    33347.88   -33347.88   -100.0
2004-04-30       0.00        0.00        0.00      NaN
2004-05-31       0.00        0.00        0.00      NaN
2004-06-30

### Q46. diff() — 연속 결제일 간격 계산

In [ ]:
#

   customerNumber paymentDate prev_pay_date  days_since_prev
0             128  2003-02-14           NaT              NaN
1             129  2003-03-15    2003-02-14             29.0
2             114  2003-05-20    2003-03-15             66.0
3             103  2003-06-05    2003-05-20             16.0
4             112  2003-06-06    2003-06-05              1.0
5             119  2004-02-02    2003-06-06            241.0
6             131  2004-07-09    2004-02-02            158.0
7             103  2004-10-19    2004-07-09            102.0
8             121  2004-11-09    2004-10-19             21.0
9             124  2004-12-18    2004-11-09             39.0


### Q47. rolling() — 3개월 이동 평균 결제액

In [ ]:
#

               amount  rolling_3m_avg
paymentDate                          
2003-02-28   14191.12    14191.120000
2003-03-31   20009.53    17100.325000
2003-04-30       0.00    11400.216667
2003-05-31    6066.78     8692.103333
2003-06-30   47213.42    17760.066667
2003-07-31       0.00    17760.066667
2003-08-31       0.00    15737.806667
2003-09-30       0.00        0.000000
2003-10-31       0.00        0.000000
2003-11-30       0.00        0.000000
2003-12-31       0.00        0.000000
2004-01-31       0.00        0.000000
2004-02-29   33347.88    11115.960000
2004-03-31       0.00    11115.960000
2004-04-30       0.00    11115.960000
2004-05-31       0.00        0.000000
2004-06-30       0.00        0.000000
2004-07-31   10223.83     3407.943333
2004-08-31       0.00     3407.943333
2004-09-30       0.00     3407.943333
2004-10-31    6066.78     2022.260000
2004-11-30   44400.50    16822.426667
2004-12-31   37281.34    29249.540000


### Q48. expanding() — 누적 최대 결제액 추적

In [ ]:
#

                daily  cumulative_max
paymentDate                          
2003-02-14   14191.12        14191.12
2003-03-15   20009.53        20009.53
2003-05-20    6066.78        20009.53
2003-06-05   14571.44        20009.53
2003-06-06   32641.98        32641.98
2004-02-02   33347.88        33347.88
2004-07-09   10223.83        33347.88
2004-10-19    6066.78        33347.88
2004-11-09   44400.50        44400.50
2004-12-18   37281.34        44400.50


---
## Part 8. 타임존

> `tz_localize()`, `tz_convert()`


### Q49. tz_localize — 타임존 없는 날짜에 타임존 부여

In [ ]:
#

타임존 없는 dtype: datetime64[ns]
UTC 적용 후: datetime64[ns, UTC]
0   2003-01-06 00:00:00+00:00
1   2003-01-09 00:00:00+00:00
2   2003-01-10 00:00:00+00:00
3   2003-01-29 00:00:00+00:00
4   2003-01-31 00:00:00+00:00
Name: orderDate, dtype: datetime64[ns, UTC]


### Q50. tz_convert — 타임존 변환 (UTC -> 서울/뉴욕)

In [ ]:
#

                        UTC                     Seoul                  New York
0 2003-01-06 09:00:00+00:00 2003-01-06 18:00:00+09:00 2003-01-06 04:00:00-05:00
1 2003-01-09 09:00:00+00:00 2003-01-09 18:00:00+09:00 2003-01-09 04:00:00-05:00
2 2003-01-10 09:00:00+00:00 2003-01-10 18:00:00+09:00 2003-01-10 04:00:00-05:00
3 2003-01-29 09:00:00+00:00 2003-01-29 18:00:00+09:00 2003-01-29 04:00:00-05:00
4 2003-01-31 09:00:00+00:00 2003-01-31 18:00:00+09:00 2003-01-31 04:00:00-05:00


---
## 최종 요약

| Part | 주제 | 핵심 함수/속성 |
|------|------|--------------|
| 1 | Timestamp & 날짜 생성 | `pd.Timestamp`, `pd.to_datetime`, `date_range`, `period_range` |
| 2 | .dt 접근자 | `.dt.year/month/day/hour/minute/second`, `.dt.day_name()`, `.dt.strftime()` |
| 3 | 시간 차이 & Timedelta | `Timedelta`, `to_timedelta`, `.dt.days`, `.dt.components` |
| 4 | DateOffset | `BDay`, `MonthEnd`, `QuarterEnd`, `CustomBusinessDay`, `DateOffset` |
| 5 | 인덱싱 & 슬라이싱 | `.loc["2003"]`, `.loc["2003-01":"2003-03"]`, `truncate()` |
| 6 | Resample & 집계 | `resample("W/ME/QE")`, `.agg()`, `.ohlc()`, `Grouper` |
| 7 | 시프트 & 롤링 | `shift()`, `diff()`, `rolling()`, `expanding()` |
| 8 | 타임존 | `tz_localize()`, `tz_convert()` |

---
> 참조: https://pandas.pydata.org/docs/user_guide/timeseries.html
